# UFT极速策略工厂类
```cpp
class WtUftStraFact : public IUftStrategyFact
```

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称的实现
 * @return const char* 返回策略工厂的名称字符串
 * 
 * 该函数返回策略工厂的名称，用于标识和管理不同的策略工厂。
 * 返回的名称是常量字符串"WtUftStraFact"，在系统中应该是唯一的。
 * 
 * @note 返回的是常量字符串指针，不需要调用者释放内存
 */
const char* WtUftStraFact::getName()
{
	return FACT_NAME;
}
```

## 枚举策略名称 enumStrategy
```cpp
/**
 * @brief 枚举策略名称的实现
 * @param cb 枚举策略名称的回调函数，每枚举到一个策略都会调用此回调
 * 
 * 该函数枚举工厂中所有可用的策略类型，通过回调函数通知调用者。
 * 当前工厂支持的策略：
 * - "SimpleUft": SimpleUft简单极速交易策略示例
 * 
 * 回调函数会被调用一次，传入以下参数：
 * - factName: 工厂名称（"WtUftStraFact"）
 * - straName: 策略名称（"SimpleUft"）
 * - isLast: 是否为最后一个策略（true，因为只有一个策略）
 * 
 * @note 如果将来添加更多策略，需要在此函数中添加更多的回调调用
 */
void WtUftStraFact::enumStrategy(FuncEnumUftStrategyCallback cb)
{
	cb(FACT_NAME, "SimpleUft", true); // 调用回调函数，传入工厂名称、策略名称和是否为最后一个策略
}
```

## 创建策略实例 createStrategy
```cpp
/**
 * @brief 创建策略实例的实现
 * @param name 策略名称，用于指定要创建的策略类型
 * @param id 策略唯一标识符，用于在系统中唯一标识该策略实例
 * @return UftStrategy* 返回创建的策略对象指针，如果策略名称不存在则返回NULL
 * 
 * 该函数根据策略名称创建对应的策略对象实例。
 * 当前支持的策略类型：
 * - "SimpleUft": 创建SimpleUft简单极速交易策略实例
 * 
 * 如果传入的策略名称不匹配任何已知策略，则返回NULL。
 * 
 * @note 调用者负责管理返回的指针，使用完毕后应通过deleteStrategy删除
 */
UftStrategy* WtUftStraFact::createStrategy(const char* name, const char* id)
{
	if(strcmp(name, "SimpleUft") == 0) // 比较策略名称是否为"SimpleUft"
	{
		return new WtUftStraDemo(id); // 创建SimpleUft策略实例并返回
	}
	return NULL;
}
```

## 删除策略实例 deleteStrategy
```cpp
/**
 * @brief 删除策略实例的实现
 * @param stra 要删除的策略对象指针
 * @return bool 删除成功返回true，失败返回false
 * 
 * 该函数删除指定的策略对象，释放相关资源。
 * 删除前会进行以下检查：
 * 1. 检查策略指针是否为空，如果为空则直接返回true（视为成功）
 * 2. 检查策略是否属于本工厂创建，通过比较策略的工厂名称
 * 3. 只有属于本工厂的策略才会被删除，其他策略返回false
 * 
 * @note 删除后策略指针将失效，调用者不应再使用该指针
 */
bool WtUftStraFact::deleteStrategy(UftStrategy* stra)
{
	if (stra == NULL)
		return true; // 如果为空，返回true（视为删除成功）

	if (strcmp(stra->getFactName(), FACT_NAME) != 0)  // 检查策略是否属于本工厂创建
		return false;

	delete stra; // 删除策略对象，调用析构函数释放资源
	return true;
}
```

# 简单极速交易策略示例类 WtUftStraDemo
```cpp
class WtUftStraDemo : public UftStrategy
```

##  成员
* **核心上下文与数据**
  * `IUftStraCtx* _ctx`：UFT策略上下文对象指针
    * 用于访问数据接口（如获取历史K线）和交易接口（如下单、撤单）
  * `WTSTickData* _last_tick`：最后接收到的Tick数据指针
    * 用于保存最新的行情快照（注：代码注释提到当前未使用，保留以备将来扩展）
* **策略配置参数**
  * `std::string _code`：合约代码。策略交易的目标合约
  * `uint32_t _secs`：订单超时时间（秒）。超过此时间未成交的订单会被自动撤销
  * `uint32_t _freq`：交易频率限制（毫秒）。两次交易操作之间的最小时间间隔
  * `int32_t _offset`：价格偏移跳数。下单价格相对于最新价的偏移量（正数向上，负数向下）
  * `double _lots`：下单数量。每次交易的手数
* **订单管理与并发控制**
  * `IDSet _orders`：订单ID集合
    * typedef std::unordered_set<uint32_t> IDSet;
    * 存储当前策略管理的所有未完成订单的本地ID
  * `SpinMutex _mtx_ords`：订单集合自旋锁互斥量
    * 用于多线程环境下保护 `_orders` 集合的并发访问安全（使用自旋锁提高极速交易场景下的性能）
  * `uint32_t _cancel_cnt`：撤销订单计数器。记录撤销的订单数量（主要用于统计和调试）
* **运行时状态与持仓**
  * `double _prev`：昨仓数量。用于记录和处理昨仓（非当日新开仓位）
  * `uint64_t _last_entry_time`：最后交易时间戳
    * 单位：微秒，用于结合 `_freq` 进行交易频率控制
  * `bool _channel_ready`：交易通道状态标记
    * `true`表示通道就绪可交易，`false`表示通道丢失或未就绪
  * `uint32_t _last_calc_time`：最后计算时间。单位：分钟，用于控制计算频率

## 基本属性

### 获取策略名称 getName
```cpp
/**
 * @brief 获取策略名称的实现
 * @return const char* 返回策略的名称字符串
 * 
 * 该函数返回策略的名称，用于标识策略类型。
 * 返回值为"UftDemoStrategy"，表示这是UFT策略示例。
 */
const char* WtUftStraDemo::getName()
{
	return "UftDemoStrategy";
}
```

### 获取工厂名称 getFactName
```cpp
/**
 * @brief 获取所属策略工厂名称的实现
 * @return const char* 返回策略所属的工厂名称字符串
 * 
 * 该函数返回策略所属的策略工厂名称，用于标识策略的来源工厂。
 * 返回值为"WtUftStraFact"，表示该策略由WtUftStraFact工厂创建。
 * 
 * @note FACT_NAME常量定义在WtUftStraFact.cpp中，值为"WtUftStraFact"
 */
const char* WtUftStraDemo::getFactName()
{
	return FACT_NAME;
}
```

## 生命周期与初始化

### 策略参数初始化 init
```cpp
/**
 * @brief 策略初始化实现
 * @param cfg 策略配置参数，包含策略运行所需的所有参数
 * @return bool 初始化成功返回true，失败返回false
 * 
 * 该函数从配置参数中加载策略运行所需的参数，包括：
 * - code: 合约代码，策略交易的合约（必需参数）
 * - second: 订单超时时间（秒），超过此时间未成交的订单会被撤销（必需参数）
 * - freq: 交易频率限制（毫秒），两次交易之间的最小时间间隔（必需参数）
 * - offset: 价格偏移跳数，下单价格相对于最新价的偏移（必需参数）
 * - lots: 下单数量，每次交易的手数（必需参数）
 * 
 * @note 如果配置参数为空或缺少必要参数，可能导致运行时错误
 */
bool WtUftStraDemo::init(WTSVariant* cfg)
{
	_code = cfg->getCString("code");
	_secs = cfg->getUInt32("second");
	_freq = cfg->getUInt32("freq");
	_offset = cfg->getUInt32("offset");

	_lots = cfg->getDouble("lots");
	return true;
}
```

### 策略初始化完成回调 on_init
**策略初始化与资源配置**。在策略加载后、正式运行前调用，用于建立上下文连接、预热数据和**注册动态参数监控**。
* **预热与订阅**
  * **数据检查**：调用 `ctx->stra_get_bars` 预读取少量 K 线数据，验证数据流是否通畅。
  * **行情订阅**：调用 `ctx->stra_sub_ticks` 订阅目标合约的实时 Tick 数据，这是驱动策略运行的源头。
* **参数监控 (UFT 特性)**
  * **关键差异**：不同于普通 HFT 策略，UFT 策略支持**盘中动态调整参数**（无需重启）。
  * **注册**：在此处会读取初始参数，并告知引擎该策略关注哪些参数的变化（以便触发 `on_params_updated`）。
* **上下文保存**
  * 将 `IUftStraCtx` 指针保存到本地，供后续逻辑调用。
```cpp
/**
 * @brief 策略初始化完成回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @note 该函数重写了UftStrategy基类的纯虚函数
 */
void WtUftStraDemo::on_init(IUftStraCtx* ctx)
```

### 参数更新回调 on_params_updated
**动态参数热更新**。当用户通过控制台或外部命令修改了策略参数时触发。
* **实时读取**
  * 从 `ctx` 中重新读取 `offset`（价格偏移）、`lots`（下单手数）、`freq`（交易频率）等核心风控与交易参数。
* **无锁/原子更新**
  * 将读取到的新值更新到策略的成员变量中。
  * **意义**：允许交易员在市场波动剧烈时实时调整下单激进程度（如增大 `offset`）或风控阈值，而无需停止策略。
* **审计日志**
  * 输出日志记录参数变更前后的值，确保操作可追溯。
```cpp
/**
 * @brief 参数更新回调实现
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_params_updated()
```

## 行情事件回调

### Tick数据回调 on_tick
这是策略的核心驱动函数。它由行情的 Tick（每一笔报价/成交）数据触发，负责执行**风控检查**、**信号计算**以及**下单交易**。
* **过滤合约**：
  * 检查传入的 `code` 是否与策略绑定的 `_code` 一致。如果不一致，直接返回。
* **未完成订单检查（串行化控制）**：
  * 检查本地维护的未完成订单集合 `_orders` 是否为空。
  * **如果不为空**：说明上一笔交易尚未结束（有未成交或未撤销的订单）。
    * 调用 `check_orders()` 检查订单是否超时。
    * **直接返回**，不进行后续的信号计算（确保策略在同一时刻只有一笔在途交易）。
* **交易通道检查**：
  * 检查 `_channel_ready` 标志。如果通道未就绪，直接返回。
* **计算频率控制**：
  * 更新 `_last_calc_time`（分钟级，虽有代码但逻辑主要被注释，主要用于重置状态）。
  * **高频流控**：计算当前时间 `now`。
  * 判断 `now - _last_entry_time`（距离上次下单的时间差）是否小于配置的 `_freq`（毫秒）。
  * 如果小于最小间隔，**直接返回**（防止频繁开仓）。
* **核心算法：理论价格计算**：
  * 获取最新价 `price`。
  * 计算**理论价格 (pxInThry)**：基于买一和卖一的**量加权平均价**。$$
\text { Theoretical Price }=\frac{\text { BidPrice }_1 \times \text { AskQty }_1+\text { AskPrice }_1 \times \text { BidQty }_1}{\text { BidQty }_1+\text { AskQty }_1}$$
  * 逻辑含义：如果卖盘量大，价格倾向下跌；如果买盘量大，价格倾向上涨。*
* **生成信号**：
  * 如果 **理论价格 > 最新价**：信号 `signal = 1`（看多）。
  * 如果 **理论价格 < 最新价**：信号 `signal = -1`（看空）。
* **执行交易**：
  * 如果 `signal != 0`：
    * 获取当前持仓 `curPos` 和 最小变动价位 `cInfo`。
    * **做多逻辑**（信号为正 且 当前空仓或无仓）：
      * 计算买入价：`最新价 + (_offset * tick)`（追价买入）。
      * 调用 `ctx->stra_buy` 下单。
      * **锁定并记录**：将返回的 `ids` 插入 `_orders` 集合，更新 `_last_entry_time`。
    * **做空逻辑**（信号为负 且 当前多仓或无仓）：
      * 计算卖出价：`最新价 - (_offset * tick)`（追价卖出）。
      * 调用 `ctx->stra_sell` 下单。
      * **锁定并记录**：将返回的 `ids` 插入 `_orders` 集合，更新 `_last_entry_time`。








```cpp
/**
 * @brief Tick数据处理回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @param code 标准合约代码，触发Tick数据的合约
 * @param newTick 新的Tick数据，包含最新的价格和成交量信息
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_tick(IUftStraCtx* ctx, const char* code, WTSTickData* newTick)
```

### K线闭合回调 on_bar
```cpp
/**
 * @brief K线闭合回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @param code 标准合约代码，触发K线数据的合约
 * @param period K线周期，如"m1"、"m5"等
 * @param times K线倍数
 * @param newBar 新的K线数据
 * 
 * 该函数在K线闭合时被调用，用于处理K线数据。
 * SimpleUft策略主要基于Tick数据，因此K线数据处理为空实现。
 * 
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_bar(IUftStraCtx* ctx, const char* code, const char* period, uint32_t times, WTSBarStruct* newBar) {}
```

## 交易回报与状态

### 订单状态回报 on_order 
负责订单状态的生命周期管理。主要用于**清理已完结的订单记录**，从而解除 `on_tick` 中的交易阻塞，允许策略进行下一次交易。
* **身份过滤**：
  * 在 `_orders` 集合中查找传入的 `localid`。
  * 如果找不到（`it == _orders.end()`），说明这不是本策略发出的订单，**直接忽略并返回**。
* **判断订单终态**：
  * 检查 `isCanceled`（是否已撤销）或者 `leftQty == 0`（剩余数量为0，即全部成交）。
  * 只要满足其中一个条件，意味着该订单生命周期结束。
* **清理与状态更新**：
  * **加锁**（`_mtx_ords.lock()`）。
  * 从 `_orders` 集合中**擦除**该订单 ID（`erase`）。
  * **更新撤单计数**：
    * 如果 `_cancel_cnt > 0`，则自减 `_cancel_cnt--`。
    * *注：此处的逻辑是，如果订单是因为策略主动撤单（超时）而结束的，在收到撤单回报时减少计数。*
  * 记录日志。
  * **解锁**。
```cpp
/**
 * @brief 订单回报回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码
 * @param isLong 是否为做多，true表示做多，false表示做空
 * @param offset 开平标志，0-开仓，1-平仓，2-平今
 * @param totalQty 订单总数量
 * @param leftQty 订单剩余数量
 * @param price 订单价格
 * @param isCanceled 是否已撤销，true表示已撤销，false表示未撤销
 * 
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_order(IUftStraCtx* ctx, uint32_t localid, const char* stdCode, bool isLong, uint32_t offset, double totalQty, double leftQty, double price, bool isCanceled)
```

### 成交回报 on_trade
```cpp
/**
 * @brief 成交回报回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @param localid 本地订单ID，用于标识订单
 * @param stdCode 标准合约代码
 * @param isLong 是否为做多，true表示做多，false表示做空
 * @param offset 开平标志，0-开仓，1-平仓，2-平今
 * @param qty 成交数量
 * @param price 成交价格
 * 
 * 该函数在订单成交时被调用，用于处理成交回报。
 * SimpleUft策略当前实现为空，可以根据需要扩展处理逻辑。
 * 
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_trade(IUftStraCtx* ctx, uint32_t localid, const char* stdCode, bool isLong, uint32_t offset, double qty, double price) {}
```

### 持仓变化回报 on_position
处理持仓更新回报。在该示例策略中，逻辑较为简单，主要用于**同步昨仓数据**和**记录日志**。
* **过滤合约**：
  * 检查 `stdCode` 是否等于策略代码 `_code`。如果不一致，返回。
* **更新昨仓**：
  * 将 `prevol`（回调传入的昨仓数量）赋值给成员变量 `_prev`。
  * *逻辑含义：策略可能在初始化或交易日切换时接收到持仓更新，此处将其记录下来（尽管在该 Demo 的交易逻辑中主要使用的是 `ctx->stra_get_position` 获取实时总持仓，`_prev` 更多作为参考或扩展用）。*
* **日志记录**：
  * 调用 `_ctx->stra_log_info` 输出该合约的昨仓数量信息。

```cpp
/**
 * @brief 持仓回报回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @param stdCode 标准合约代码
 * @param isLong 是否为多头，true表示多头，false表示空头
 * @param prevol 变化前的持仓量（昨仓）
 * @param preavail 变化前的可用持仓量（可用昨仓）
 * @param newvol 变化后的持仓量（今仓）
 * @param newavail 变化后的可用持仓量（可用今仓）
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_position(IUftStraCtx* ctx, const char* stdCode, bool isLong, double prevol, double preavail, double newvol, double newavail)
```

### 委托结果回报 on_entrust
这是**委托回报**回调函数。它不同于订单回报（`on_order`），主要用于反馈“发单请求”本身是否成功被交易接口或柜台接收。如果发单请求在本地API校验失败或被网关拒绝，会触发此回调。此函数主要用于**快速清理失败的订单记录**。
* **判断委托结果**：
  * 检查参数 `bSuccess`。
  * 如果为 `true`，表示委托发送成功（但未成交），不做处理，后续状态由 `on_order` 接管。
* **处理失败委托**（`!bSuccess`）：
  * **查找订单**：在本地维护的未完成订单集合 `_orders` 中查找对应的 `localid`。
  * **清理记录**：
    * 如果找到该 ID，直接从 `_orders` 集合中**移除**（`erase`）。
    * *逻辑含义：既然委托发送失败，说明订单根本不存在于交易所队列中，后续也不会有成交或撤单回报，因此必须立即清除本地记录，否则策略会一直误以为有“未完成订单”而阻塞后续交易（即死锁）。*
```cpp
/**
 * @brief 委托回报回调实现
 * @param localid 本地订单ID，用于标识订单
 * @param bSuccess 委托是否成功，true表示成功，false表示失败
 * @param message 委托结果消息
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_entrust(uint32_t localid, bool bSuccess, const char* message)
```

## 交易通道事件

### 交易通道就绪回调 on_channel_ready
这是**交易通道就绪**的回调。当策略与交易柜台连接建立完成并准备好交易时触发。此函数不仅是一个简单的状态标记，还包含重要的**状态清洗与断点恢复**逻辑，确保策略在“干净”的环境下启动。
* **遗留订单检查（安全机制）**：
  * 调用 `_ctx->stra_get_undone` 获取该合约当前在柜台的实际挂单数量 `undone`。
  * 判断条件：如果 `undone != 0` 且 本地 `_orders` 集合为空。
    * *场景含义：这通常发生在策略崩溃重启或断线重连后。柜台还有之前的挂单，但策略内存重启后丢失了这些订单ID，导致“失控”。*
* **清洗失控订单**：
  * **全部撤单**：记录日志并调用 `_ctx->stra_cancel_all` 撤销该合约所有在途订单。
  * **接管撤单状态**：
    * 获取撤单接口返回的订单 ID 列表 `ids`。
    * 遍历这些 `ids`，将其**强行插入**本地 `_orders` 集合。
    * *目的：虽然发起了撤单，但必须等待柜台返回“已撤单”的 `on_order` 回调。如果不加入 `_orders`，`on_order` 会因为识别不出这些 ID 而忽略它们，导致状态无法闭环。*
  * **更新计数**：增加 `_cancel_cnt`。
* **开启交易许可**：
  * 将 `_channel_ready` 标记设为 `true`。
  * 允许 `on_tick` 函数开始执行核心交易逻辑。
```cpp
/**
 * @brief 交易通道就绪回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_channel_ready(IUftStraCtx* ctx)
```

### 交易通道丢失回调 on_channel_lost
```cpp
/**
 * @brief 交易通道丢失回调实现
 * @param ctx UFT策略上下文对象，提供数据访问和交易执行接口
 * 
 * 该函数在交易通道丢失时被调用，表示无法进行交易。
 * 主要功能：
 * 1. 设置通道丢失标志，禁止策略执行交易
 * 2. 等待通道恢复后再继续交易
 * 
 * @note 该函数重写了UftStrategy基类的虚函数
 */
void WtUftStraDemo::on_channel_lost(IUftStraCtx* ctx)
{
	_channel_ready = false;
}
```

## 订单检查与超时处理 check_orders
这是**订单超时管理**逻辑。它不是一个回调函数，而是由 `on_tick` 驱动的主动检查函数。其作用是防止订单长时间挂在盘口不成交（“占款”或“占单”），实施**Time-To-Live (TTL)** 策略。
* **前置校验**：
  * 检查 `_orders` 是否不为空（有未完成单）。
  * 检查 `_last_entry_time` 是否有效（不为 `UINT64_MAX`）。
* **超时判断**：
  * 获取当前系统时间 `now`（微秒级）。
  * 计算时间差：`now - _last_entry_time`。
  * 判断差值是否大于设定的超时阈值 `_secs * 1000`（秒转微秒）。
* **执行超时撤单**：
  * 如果判断为超时：
    * **加锁**（`_mtx_ords.lock()`）。
    * **批量撤单**：
      * 遍历 `_orders` 集合中的每一个 `localid`。
      * 调用 `_ctx->stra_cancel(localid)` 发送撤单指令。
      * 增加撤单计数器 `_cancel_cnt`。
      * 记录“Order expired”日志。
    * **解锁**。
  * *注意：此处只负责“发令”撤单，不负责从 `_orders` 移除 ID。ID 的移除操作会由 `on_order` 在收到柜台确认撤单后执行。*
```cpp
/**
 * @brief 检查订单状态的实现
 * @note 该函数是私有函数，仅在策略内部调用
 */
void WtUftStraDemo::check_orders()
```